In [78]:
import torch
import torch.nn as nn

In [79]:
text = "hello world hello machine learning pytorch is powerful learning is fun"

In [80]:
# -------------------------
# 1. Character vocabulary
# -------------------------

chars = sorted(set(text))

stoi = {ch:i for i, ch in enumerate(chars) }
itos = {i:ch for ch, i in stoi.items()}


print("stoi:", stoi)
print("itos:", itos)

stoi: {' ': 0, 'a': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'l': 9, 'm': 10, 'n': 11, 'o': 12, 'p': 13, 'r': 14, 's': 15, 't': 16, 'u': 17, 'w': 18, 'y': 19}
itos: {0: ' ', 1: 'a', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'l', 10: 'm', 11: 'n', 12: 'o', 13: 'p', 14: 'r', 15: 's', 16: 't', 17: 'u', 18: 'w', 19: 'y'}


In [94]:
# -------------------------
# 2. Prepare data
# -------------------------

BLOCK_SIZE = 4
D_MODEL = 64
VOCAB_SIZE = len(chars)
HIDDEN_SIZE = 128

data = torch.tensor([stoi[x] for x in text])


print(data.size())
print("".join(itos[x.item()] for x in data[0:4]) + "-> " + "".join(itos[data[4].item()]))

X =  []
Y =  []

for i in range(len(data) - BLOCK_SIZE):
    X.append( data[i: i + BLOCK_SIZE] )
    Y.append(data[i + BLOCK_SIZE])

X = torch.stack(X)
Y = torch.stack(Y)


print("vocab_size:", VOCAB_SIZE)
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("X:", X)
print("Y:", Y)

torch.Size([70])
hell-> o
vocab_size: 20
X shape: torch.Size([66, 4])
Y shape: torch.Size([66])
X: tensor([[ 7,  4,  9,  9],
        [ 4,  9,  9, 12],
        [ 9,  9, 12,  0],
        [ 9, 12,  0, 18],
        [12,  0, 18, 12],
        [ 0, 18, 12, 14],
        [18, 12, 14,  9],
        [12, 14,  9,  3],
        [14,  9,  3,  0],
        [ 9,  3,  0,  7],
        [ 3,  0,  7,  4],
        [ 0,  7,  4,  9],
        [ 7,  4,  9,  9],
        [ 4,  9,  9, 12],
        [ 9,  9, 12,  0],
        [ 9, 12,  0, 10],
        [12,  0, 10,  1],
        [ 0, 10,  1,  2],
        [10,  1,  2,  7],
        [ 1,  2,  7,  8],
        [ 2,  7,  8, 11],
        [ 7,  8, 11,  4],
        [ 8, 11,  4,  0],
        [11,  4,  0,  9],
        [ 4,  0,  9,  4],
        [ 0,  9,  4,  1],
        [ 9,  4,  1, 14],
        [ 4,  1, 14, 11],
        [ 1, 14, 11,  8],
        [14, 11,  8, 11],
        [11,  8, 11,  6],
        [ 8, 11,  6,  0],
        [11,  6,  0, 13],
        [ 6,  0, 13, 19],
        [ 0, 13, 

In [95]:
# -------------------------
# 3. RNNModel 
# -------------------------
class RNNModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL
        )

        self.rnn = nn.RNN(
            input_size=D_MODEL,
            hidden_size=HIDDEN_SIZE,
            batch_first=True
        )


        self.fc = nn.Linear(
            HIDDEN_SIZE,
            VOCAB_SIZE
        )

    def forward(self, x):
        x = self.embedding(x)
        x, hidden = self.rnn(x)
        logits = self.fc(x)
        return logits

torch.manual_seed(42)

model  = RNNModel()

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)


Trainable parameters: 28692


In [96]:
# -------------------------
# 4. Training + accuracy
# -------------------------


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

for step in range(1000):

    logits = model(X)

    loss = nn.functional.cross_entropy(
        logits[:, -1, :],
        Y
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())

0 3.0132522583007812
100 0.0631575658917427
200 0.06314437091350555
300 0.06308815628290176
400 0.06307279318571091
500 0.06306540966033936
600 0.0630565881729126
700 0.06306842714548111
800 0.06304819881916046
900 0.06313654035329819


In [99]:
# -------------------------
# 5. Generate
# -------------------------
torch.manual_seed(1337)

context = torch.tensor([[stoi[c] for c in "learning"]])
result = "learning"

for _ in range(20):

    logits = model(context)

    # Last timestep
    logits = logits[:, -1, :]

    probs = torch.softmax(logits, dim=-1)

    next_char = torch.multinomial(
        probs,
        num_samples=1
    )

    result += itos[next_char.item()]

    context = torch.cat([context, next_char], dim=1)[:, -BLOCK_SIZE:]

    print("context", context)

print("\nGenerated:")
print(result)

context tensor([[ 8, 11,  6,  0]])
context tensor([[11,  6,  0, 13]])
context tensor([[ 6,  0, 13, 19]])
context tensor([[ 0, 13, 19, 16]])
context tensor([[13, 19, 16, 12]])
context tensor([[19, 16, 12, 14]])
context tensor([[16, 12, 14,  2]])
context tensor([[12, 14,  2,  7]])
context tensor([[14,  2,  7,  0]])
context tensor([[2, 7, 0, 8]])
context tensor([[ 7,  0,  8, 15]])
context tensor([[ 0,  8, 15,  0]])
context tensor([[ 8, 15,  0,  5]])
context tensor([[15,  0,  5, 17]])
context tensor([[ 0,  5, 17, 11]])
context tensor([[ 5, 17, 11,  6]])
context tensor([[17, 11,  6,  0]])
context tensor([[11,  6,  0, 13]])
context tensor([[ 6,  0, 13, 19]])
context tensor([[ 0, 13, 19, 16]])

Generated:
learning pytorch is fung pyt
